In [1]:
import pandas as pd

# Clean patient_feedback.csv for the Patient Experience Office analysis.
# Everything from here on reads from / writes to df_clean
df_raw = pd.read_csv("patient_feedback.csv")
df_clean = df_raw.copy()

print(f"Raw rows: {len(df_clean):,}")
df_clean.head()

Raw rows: 300,000


,feedback_id,appointment_id,patient_id,department_id,rating,complaint_category,comment,feedback_date,patient_satisfaction_after_visit
0,F000000001,A000052897,P0052751,D038,4,Waiting,I waited much longer than expected.,2024-01-29,Satisfied
1,F000000002,A000679409,P0071824,D007,3,Wait Time,The wait was too long.,2025-05-14,At Risk
2,F000000003,A000724365,P0085277,D045,3,Communication,Instructions were unclear.,2025-11-11,At Risk
3,F000000004,A000303615,P0029668,D039,3,Communication,Communication could be better.,2023-11-24,At Risk
4,F000000005,A000213081,P0014845,D027,3,Wait Time,The wait was too long.,2024-02-11,At Risk


In [2]:
# 1. patient_id -- fix inconsistent formatting
#    'P0052751'        (8 chars,  281,478 rows)
#    'PAT-0089415'     (11 chars,  18,316 rows)
#    'PAT-AT-0088888'  (14 chars,     206 rows)
# Standardize all patient IDs to: P + 7 digits (8 characters total)

df_clean["patient_id"] = (
    df_clean["patient_id"]
    .str.extract(r"(\d+)$")[0]      # pull trailing digits regardless of prefix
    .str.zfill(7)                    # pad to a consistent width
)
df_clean["patient_id"] = "P" + df_clean["patient_id"]

print("patient_id format check after cleaning:")
print(df_clean["patient_id"].str.len().value_counts())

patient_id format check after cleaning:
patient_id
8    300000
Name: count, dtype: int64


In [3]:
# 2. feedback_date -- 4 formats mixed in one column:
#    'YYYY-MM-DD'   (273,482 rows)
#    'YYYY/MM/DD'   (  4,824 rows)
#    'MM/DD/YYYY'   (  4,728 rows)
#    'DD-Mon-YYYY'  (  4,915 rows)
# Convert each format in turn, checking only the dates that have not already been converted. 
# Blank cells in the original data (12,051 rows) are kept as missing values.

date_formats = ["%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y", "%d-%b-%Y"]
parsed = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns]")
for fmt in date_formats:
    still_missing = parsed.isna()
    attempt = pd.to_datetime(df_clean.loc[still_missing, "feedback_date"], format=fmt, errors="coerce")
    parsed.loc[still_missing] = attempt
df_clean["feedback_date"] = parsed

failed_to_parse = df_clean["feedback_date"].isna().sum() - df_raw["feedback_date"].isna().sum()
# 0 = every non-blank value converted

df_clean["flag_no_date"] = df_clean["feedback_date"].isna()
print(f"Missing feedback_date (blank in source): {df_clean['flag_no_date'].sum():,}")

Missing feedback_date (blank in source): 12,051


In [4]:
# 3. complaint_category -- 21 raw values become 6 real categories once
#    you fix casing, abbreviations, and a trailing-space typo.
#    'none' is treated as "no category" (mapped to missing),
#    same as a true missing value, not its own category.

category_map = {
    "billing": "Billing", "billing issue": "Billing", "bill": "Billing", "payment": "Billing",
    "communication": "Communication", "communication issue": "Communication",
    "comm": "Communication", "info": "Communication",
    "provider": "Provider", "provider issue": "Provider", "doctor": "Provider", "clinician": "Provider",
    "facilities": "Facilities", "facility": "Facilities", "building": "Facilities",
    "wait time": "Wait Time", "long wait": "Wait Time", "waiting": "Wait Time", "wait": "Wait Time",
    "none": None,
}
df_clean["complaint_category"] = (
    df_clean["complaint_category"].str.strip().str.lower().map(category_map)
)

print("complaint_category value counts after mapping (NaN + mapped-None are both missing):")
print(df_clean["complaint_category"].value_counts(dropna=False))
print(f"\nTotal missing complaint_category: {df_clean['complaint_category'].isna().sum():,}")

complaint_category value counts after mapping (NaN + mapped-None are both missing):
complaint_category
NaN              137902
Wait Time         66348
Communication     54113
Billing           18245
Provider          14049
Facilities         6983
None               2360
Name: count, dtype: int64

Total missing complaint_category: 140,262


In [5]:
# 4. # rating -- check the values that appear in the data before deciding which
#     ones are valid. Ratings from 1 to 5 account for 298,200 of 300,000 rows
#     (99.4%), suggesting this is the intended scale. The values -1, 0, 6, 7,
#     and 99 appear only in small numbers and are likely data entry errors or
#     placeholder values rather than real ratings.


print(df_clean["rating"].value_counts().sort_index())

valid_ratings = [1, 2, 3, 4, 5]
df_clean["flag_invalid_rating"] = ~df_clean["rating"].isin(valid_ratings)

print(f"\nInvalid rating rows flagged: {df_clean['flag_invalid_rating'].sum():,}")

rating
-1        364
 0        400
 1       1731
 2      21396
 3      91877
 4     130737
 5      52459
 6        350
 7        349
 99       337
Name: count, dtype: int64

Invalid rating rows flagged: 1,800


In [6]:
# 5. comment -- flag missing (can't be theme-coded)
# A missing comment can't be classified into a theme at all, so it's
# excluded from the theme / root-cause analysis.

df_clean["flag_no_comment"] = df_clean["comment"].isna()

print(f"Missing-comment rows flagged: {df_clean['flag_no_comment'].sum():,}")

Missing-comment rows flagged: 7,494


In [7]:
# 6. department_id -- flag missing
# Needed for the department hotspot / join-based analysis. Ratings/comments
# can still be valid even if dept is missing, but the row can't be used in
# any department-level breakdown.

df_clean["flag_no_department"] = df_clean["department_id"].isna()

print(f"Missing-department rows flagged: {df_clean['flag_no_department'].sum():,}")

Missing-department rows flagged: 884


In [8]:
# 7. Combined "clean" flag
# A row is usable for the core analysis only if the rating is valid, the
# comment exists, AND the department is known. Missing feedback_date does
# NOT disqualify a row here -- it just can't be used in time-trend cuts,
# so it's tracked separately (flag_no_date) instead of forcing an exclusion.

df_clean["is_clean"] = ~(
    df_clean["flag_invalid_rating"] | df_clean["flag_no_comment"] | df_clean["flag_no_department"]
)

print(f"Clean rows for analysis: {df_clean['is_clean'].sum():,}")
print(f"Excluded rows (either flag): {(~df_clean['is_clean']).sum():,}")

Clean rows for analysis: 289,889
Excluded rows (either flag): 10,111


In [9]:
# 8. Derive At-Risk label from valid ratings
# (business rule: rating <= 3 -> At Risk, rating >= 4 -> Satisfied)
# Sanity check: the dataset already ships a patient_satisfaction_after_visit
# label -- confirm our derived flag agrees with it on every clean row before
# trusting it downstream.

df_clean["at_risk"] = df_clean["rating"] <= 3

clean_rows = df_clean.loc[df_clean["is_clean"]]
agreement = (
    clean_rows["at_risk"] == (clean_rows["patient_satisfaction_after_visit"] == "At Risk")
).mean()
print(f"at_risk (derived) vs patient_satisfaction_after_visit (given) agreement: {agreement:.2%}")

at_risk_rate = clean_rows["at_risk"].mean()
print(f"At-Risk rate on clean data: {at_risk_rate:.1%}")

at_risk (derived) vs patient_satisfaction_after_visit (given) agreement: 100.00%
At-Risk rate on clean data: 38.6%


In [10]:
# 9. Audit summary

summary = pd.DataFrame({
    "step": [
        "Raw rows",
        "Invalid rating (removed)",
        "Missing comment (removed)",
        "Missing department_id (removed)",
        "Missing feedback_date (kept, tracked only)",
        "Clean rows remaining",
    ],
    "count": [
        len(df_clean),
        df_clean["flag_invalid_rating"].sum(),
        df_clean["flag_no_comment"].sum(),
        df_clean["flag_no_department"].sum(),
        df_clean["flag_no_date"].sum(),
        df_clean["is_clean"].sum(),
    ],
})
print("Audit summary:")
print(summary.to_string(index=False))

Audit summary:
                                      step  count
                                  Raw rows 300000
                  Invalid rating (removed)   1800
                 Missing comment (removed)   7494
           Missing department_id (removed)    884
Missing feedback_date (kept, tracked only)  12051
                      Clean rows remaining 289889


In [11]:
# 10. Pre-merge check: appointment_id fan-out
# patient_feedback joins to appointments on appointment_id. If any
# appointment_id has more than one feedback row, an inner join will
# duplicate that appointment's other fields across each feedback entry --
# not wrong, but worth knowing about before interpreting row counts downstream.

dupe_appts = df_clean["appointment_id"].duplicated().sum()
print(f"appointment_id values with more than one feedback row: {dupe_appts:,}")
print("(Expected if patients can submit multiple feedback entries per visit; "
      "if not expected, investigate these before merging.)")

appointment_id values with more than one feedback row: 1,082
(Expected if patients can submit multiple feedback entries per visit; if not expected, investigate these before merging.)


In [12]:
# 11. Finalize df_clean for merging
# Keep only the clean rows and the columns needed downstream. All messy
# raw columns have been overwritten in place, so no separate "_clean"
# suffix or rename step is needed -- df_clean already reflects final values.

df_clean = df_clean.loc[df_clean["is_clean"], [
    "feedback_id",
    "appointment_id",
    "patient_id",
    "department_id",
    "rating",
    "at_risk",
    "complaint_category",
    "comment",
    "feedback_date",
    "patient_satisfaction_after_visit",
]].reset_index(drop=True)

print("df_clean shape:", df_clean.shape)
print(df_clean.dtypes)
df_clean.head()

df_clean shape: (289889, 10)
feedback_id                                 object
appointment_id                              object
patient_id                                  object
department_id                               object
rating                                       int64
at_risk                                       bool
complaint_category                          object
comment                                     object
feedback_date                       datetime64[ns]
patient_satisfaction_after_visit            object
dtype: object


,feedback_id,appointment_id,patient_id,department_id,rating,at_risk,complaint_category,comment,feedback_date,patient_satisfaction_after_visit
0,F000000001,A000052897,P0052751,D038,4,False,Wait Time,I waited much longer than expected.,2024-01-29,Satisfied
1,F000000002,A000679409,P0071824,D007,3,True,Wait Time,The wait was too long.,2025-05-14,At Risk
2,F000000003,A000724365,P0085277,D045,3,True,Communication,Instructions were unclear.,2025-11-11,At Risk
3,F000000004,A000303615,P0029668,D039,3,True,Communication,Communication could be better.,2023-11-24,At Risk
4,F000000005,A000213081,P0014845,D027,3,True,Wait Time,The wait was too long.,2024-02-11,At Risk


In [13]:
# ============================================================
# Shared helper: parse a column with mixed date formats
# (same 4-format pattern seen in feedback_date -- appears again in
# appointments.csv, patients.csv, and claims.csv, so it's pulled out once here)
# ============================================================

def parse_multi_format(s, formats=("%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y", "%d-%b-%Y")):
    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in formats:
        still_missing = parsed.isna()
        attempt = pd.to_datetime(s[still_missing], format=fmt, errors="coerce")
        parsed[still_missing] = attempt
    return parsed

In [14]:
# ============================================================
# Clean appointments.csv
# ============================================================
appointments_raw = pd.read_csv("appointments.csv")
appointments_clean = appointments_raw.copy()

print(f"Raw rows: {len(appointments_clean):,}")
appointments_clean.head()

Raw rows: 1,012,000


,appointment_id,patient_id,provider_id,department_id,scheduled_date,appointment_date,visit_type,wait_days,status,future_no_show_flag
0,A000000001,P0082338,PR001466,D006,2024-11-14,2024-11-14,Urgent,0,Completed,0
1,A000000002,P0003609,PR001137,D031,09-Jan-2024,2024-01-24,Procedure,15,Completed,0
2,A000000003,P0060334,PR001536,D035,2024-10-09,2024-10-09,New Consult,0,Completed,0
3,A000000004,PAT-0096218,PR001172,D028,2023-10-28,2023-11-07,Urgent,10,Completed,0
4,A000000005,P0093714,PR001723,D044,2024/06/13,2024-07-16,Follow-up,33,No Show,1


In [15]:
# 1. Exact duplicate rows -- 12,000 appointment_ids appear TWICE with every
#    single column identical. Unlike patient_feedback (where a repeated
#    appointment_id is a legitimate second feedback submission), this is the
#    same appointment record duplicated outright - dropped.

n_before = len(appointments_clean)
appointments_clean = appointments_clean.drop_duplicates().reset_index(drop=True)
n_dupes_dropped = n_before - len(appointments_clean)

print(f"Exact duplicate rows dropped: {n_dupes_dropped:,}")
print(f"Rows remaining: {len(appointments_clean):,}")

Exact duplicate rows dropped: 12,000
Rows remaining: 1,000,000


In [16]:
# 2. patient_id -- same 3-format inconsistency as patient_feedback.csv.
#    Standardize to P + 7 digits so it joins cleanly to patients_clean / df_clean.

appointments_clean["patient_id"] = (
    appointments_clean["patient_id"].str.extract(r"(\d+)$")[0].str.zfill(7)
)
appointments_clean["patient_id"] = "P" + appointments_clean["patient_id"]

print("patient_id format check after cleaning:")
print(appointments_clean["patient_id"].str.len().value_counts())

patient_id format check after cleaning:
patient_id
8    1000000
Name: count, dtype: int64


In [17]:
# 3. scheduled_date / appointment_date -- same 4 mixed date formats as
#    feedback_date. convert both, then RECOMPUTE wait_days from the converted
#    dates instead of trusting the raw column: raw wait_days disagrees with
#    (appointment_date - scheduled_date) on exactly 8,087 rows, and every one
#    of those is the full set of negative raw values (a negative wait time
#    is impossible). Recomputing from the actual dates is more reliable than
#    keeping a derived column that was corrupted for a known subset of rows.

appointments_clean["scheduled_date"] = parse_multi_format(appointments_clean["scheduled_date"])
appointments_clean["appointment_date"] = parse_multi_format(appointments_clean["appointment_date"])

recomputed_wait = (appointments_clean["appointment_date"] - appointments_clean["scheduled_date"]).dt.days
appointments_clean["flag_wait_days_corrected"] = recomputed_wait != appointments_clean["wait_days"]
appointments_clean["wait_days"] = recomputed_wait

print(f"wait_days rows corrected from recomputed dates: {appointments_clean['flag_wait_days_corrected'].sum():,}")
print(appointments_clean["wait_days"].describe())

wait_days rows corrected from recomputed dates: 8,000
count    1000000.000000
mean          23.358920
std           14.612684
min            0.000000
25%           11.000000
50%           23.000000
75%           35.000000
max           73.000000
Name: wait_days, dtype: float64


In [18]:
# 4. status -- 17 raw values become 4 real statuses once you fix
#    casing, abbreviations, and spelling variants 

status_map = {
    "completed": "Completed", "complete": "Completed", "done": "Completed",
    "no show": "No Show", "no-show": "No Show", "noshow": "No Show",
    "cancelled": "Cancelled", "canceled": "Cancelled", "cancel": "Cancelled",
    "rescheduled": "Rescheduled", "re-scheduled": "Rescheduled", "moved": "Rescheduled",
}
appointments_clean["status"] = (
    appointments_clean["status"].str.strip().str.lower().map(status_map)
)

print("status value counts after mapping:")
print(appointments_clean["status"].value_counts(dropna=False))

status value counts after mapping:
status
Completed      769656
No Show        112869
Cancelled       70814
Rescheduled     40676
NaN              5985
Name: count, dtype: int64


In [19]:
# 5. Flag missing provider_id / department_id / status.
# These are all sparse (<0.6% each) but each disqualifies the row from a
# different downstream cut (provider-level, department-level, status-based).

appointments_clean["flag_no_provider"] = appointments_clean["provider_id"].isna()
appointments_clean["flag_no_department"] = appointments_clean["department_id"].isna()
appointments_clean["flag_no_status"] = appointments_clean["status"].isna()

print(f"Missing provider_id: {appointments_clean['flag_no_provider'].sum():,}")
print(f"Missing department_id: {appointments_clean['flag_no_department'].sum():,}")
print(f"Missing status: {appointments_clean['flag_no_status'].sum():,}")

Missing provider_id: 3,929
Missing department_id: 2,935
Missing status: 5,985


In [20]:
# 6. Combined "clean" flag + audit summary
# A row is usable for the core analysis only if provider_id, department_id,
# AND status are all known. wait_days being corrected does NOT disqualify a
# row -- it's tracked separately (flag_wait_days_corrected) since it was fixed,
# not lost.

appointments_clean["is_clean"] = ~(
    appointments_clean["flag_no_provider"]
    | appointments_clean["flag_no_department"]
    | appointments_clean["flag_no_status"]
)

summary = pd.DataFrame({
    "step": [
        "Raw rows",
        "Exact duplicate rows (dropped)",
        "Missing provider_id (excluded)",
        "Missing department_id (excluded)",
        "Missing status (excluded)",
        "wait_days corrected (kept, tracked only)",
        "Clean rows remaining",
    ],
    "count": [
        len(appointments_raw),
        n_dupes_dropped,
        appointments_clean["flag_no_provider"].sum(),
        appointments_clean["flag_no_department"].sum(),
        appointments_clean["flag_no_status"].sum(),
        appointments_clean["flag_wait_days_corrected"].sum(),
        appointments_clean["is_clean"].sum(),
    ],
})
print("Audit summary:")
print(summary.to_string(index=False))

Audit summary:
                                    step   count
                                Raw rows 1012000
          Exact duplicate rows (dropped)   12000
          Missing provider_id (excluded)    3929
        Missing department_id (excluded)    2935
               Missing status (excluded)    5985
wait_days corrected (kept, tracked only)    8000
                    Clean rows remaining  987211


In [21]:
# 7. Finalize appointments_clean for merging

appointments_clean = appointments_clean.loc[appointments_clean["is_clean"], [
    "appointment_id",
    "patient_id",
    "provider_id",
    "department_id",
    "scheduled_date",
    "appointment_date",
    "visit_type",
    "wait_days",
    "status",
    "future_no_show_flag",
    "flag_wait_days_corrected",
]].reset_index(drop=True)

print("appointments_clean shape:", appointments_clean.shape)
print(appointments_clean.dtypes)
appointments_clean.head()

appointments_clean shape: (987211, 11)
appointment_id                      object
patient_id                          object
provider_id                         object
department_id                       object
scheduled_date              datetime64[ns]
appointment_date            datetime64[ns]
visit_type                          object
wait_days                            int64
status                              object
future_no_show_flag                  int64
flag_wait_days_corrected              bool
dtype: object


,appointment_id,patient_id,provider_id,department_id,scheduled_date,appointment_date,visit_type,wait_days,status,future_no_show_flag,flag_wait_days_corrected
0,A000000001,P0082338,PR001466,D006,2024-11-14,2024-11-14,Urgent,0,Completed,0,False
1,A000000002,P0003609,PR001137,D031,2024-01-09,2024-01-24,Procedure,15,Completed,0,False
2,A000000003,P0060334,PR001536,D035,2024-10-09,2024-10-09,New Consult,0,Completed,0,False
3,A000000004,P0096218,PR001172,D028,2023-10-28,2023-11-07,Urgent,10,Completed,0,False
4,A000000005,P0093714,PR001723,D044,2024-06-13,2024-07-16,Follow-up,33,No Show,1,False


In [22]:
# ============================================================
# Clean patients.csv
# ============================================================
patients_raw = pd.read_csv("patients.csv")
patients_clean = patients_raw.copy()

print(f"Raw rows: {len(patients_clean):,}")
patients_clean.head()

Raw rows: 105,000


,patient_id,first_name,last_name,dob,gender,region,postal_code,insurance_type,chronic_diabetes,chronic_hypertension,registration_date
0,P0000001,Sam,Patel,1982-08-09,Female,Other,J6P 9O0,Employer,False,False,2024-11-30
1,P0000002,Riley,Garcia,1977-02-13,Male,Other,X7L 8B9,Employer,False,True,2019-06-07
2,P0000003,Jamie,Singh,1971-02-25,Female,Mississauga,K4A 7O0,Private,False,False,2024-06-27
3,P0000004,Taylor,Kim,1946-08-15,Male,Oakville,L0K 5R0,Employer,False,False,2023-12-07
4,P0000005,Alex,Patel,1976-11-10,f,Brampton,X0I 0H4,Public,False,True,2019/09/21


In [23]:
# 1. patient_id -- fix inconsistent formatting
#    'P0000002'    (8 chars,  100,000 rows)
#    'PAT-0000002' (11 chars,   5,000 rows)
# Standardize to P + 7 digits, same as the other tables.

patients_clean["patient_id"] = (
    patients_clean["patient_id"].str.extract(r"(\d+)$")[0].str.zfill(7)
)
patients_clean["patient_id"] = "P" + patients_clean["patient_id"]

print("patient_id format check after cleaning:")
print(patients_clean["patient_id"].str.len().value_counts())
print(f"\nDuplicate patient_id AFTER standardizing: {patients_clean['patient_id'].duplicated().sum():,}")

patient_id format check after cleaning:
patient_id
8    105000
Name: count, dtype: int64

Duplicate patient_id AFTER standardizing: 5,000


In [24]:
# 2. Investigate the 5,000 duplicate patient_ids identified above.
#    Comparing patient details such as name, date of birth, region, postal code,
#    insurance type, chronic condition indicators, and registration date showed
#    that the duplicate records match the original records exactly. The duplicate
#    entries use a different ID format (starting with 'PAT-') and names in
#    uppercase, but otherwise contain the same information. These records appear
#    to be repeated entries added at the end of the dataset, so they can be
#    safely removed while keeping the first occurrence of each patient.

n_before = len(patients_clean)
patients_clean = patients_clean.drop_duplicates(subset="patient_id", keep="first").reset_index(drop=True)
n_duplicate_patients_dropped = n_before - len(patients_clean)

print(f"Duplicate patient records dropped: {n_duplicate_patients_dropped:,}")
print(f"Unique patients remaining: {len(patients_clean):,}")
assert patients_clean["patient_id"].is_unique

Duplicate patient records dropped: 5,000
Unique patients remaining: 100,000


In [25]:
# 3. dob / registration_date -- same 4 mixed date formats seen elsewhere.

patients_clean["dob"] = parse_multi_format(patients_clean["dob"])
patients_clean["registration_date"] = parse_multi_format(patients_clean["registration_date"])

patients_clean["flag_no_dob"] = patients_clean["dob"].isna()
print(f"Missing dob: {patients_clean['flag_no_dob'].sum():,}")

age_years = (pd.Timestamp("today") - patients_clean["dob"]).dt.days / 365.25
print("\nImplied age range (sanity check):")
print(age_years.describe())

Missing dob: 1,792

Implied age range (sanity check):
count    98208.000000
mean        53.447365
std         21.970415
min         15.520876
25%         34.405886
50%         53.396304
75%         72.452430
max         91.509925
Name: dob, dtype: float64


In [26]:
# 4. gender -- 14 raw values become Female / Male / Other / Unknown
#    once you fix casing, abbreviations ('F'/'M'), and synonyms ('Woman'/'Man').
#    Stray leading/trailing spaces on some rows (' f ', ' m ') are stripped first.

gender_map = {
    "female": "Female", "f": "Female", "woman": "Female",
    "male": "Male", "m": "Male", "man": "Male",
    "other": "Other",
    "unknown": "Unknown",
}
patients_clean["gender"] = (
    patients_clean["gender"].str.strip().str.lower().map(gender_map)
)
patients_clean["flag_no_gender"] = patients_clean["gender"].isna()

print("gender value counts after mapping:")
print(patients_clean["gender"].value_counts(dropna=False))
print(f"\nMissing gender (blank in source): {patients_clean['flag_no_gender'].sum():,}")

gender value counts after mapping:
gender
Female     49953
Male       48049
Other       1889
NaN           64
Unknown       45
Name: count, dtype: int64

Missing gender (blank in source): 64


In [27]:
# 5. postal_code -- flag placeholder/sentinel values as missing, and
#    normalize the no-space format ('A1A1A1') to standard 'A1A 1A1' spacing.
#    Sentinels found: 'UNKNOWN' (328), '00000' (317), '123 ABC' (309 -- an
#    obvious placeholder, not a real Canadian postal code).

invalid_postal_sentinels = ["UNKNOWN", "00000", "123 ABC"]
patients_clean["postal_code"] = patients_clean["postal_code"].replace(invalid_postal_sentinels, None)

no_space_mask = patients_clean["postal_code"].str.match(r"^[A-Za-z]\d[A-Za-z]\d[A-Za-z]\d$", na=False)
patients_clean.loc[no_space_mask, "postal_code"] = (
    patients_clean.loc[no_space_mask, "postal_code"].str[:3]
    + " "
    + patients_clean.loc[no_space_mask, "postal_code"].str[3:]
)

patients_clean["flag_no_postal_code"] = patients_clean["postal_code"].isna()
print(f"Missing/invalid postal_code: {patients_clean['flag_no_postal_code'].sum():,}")

Missing/invalid postal_code: 3,661


In [28]:
# 6. Audit summary + finalize patients_clean
#    No rows need to be excluded here -- patient_id is complete and unique
#    after dedup, which is the only required join key. dob / gender /
#    postal_code issues are tracked via flag columns, not required for the join.

summary = pd.DataFrame({
    "step": [
        "Raw rows",
        "Duplicate patient records (dropped)",
        "Missing dob (kept, tracked only)",
        "Missing/Unknown gender (kept, tracked only)",
        "Missing/invalid postal_code (kept, tracked only)",
        "Unique patients remaining",
    ],
    "count": [
        len(patients_raw),
        n_duplicate_patients_dropped,
        patients_clean["flag_no_dob"].sum(),
        patients_clean["flag_no_gender"].sum(),
        patients_clean["flag_no_postal_code"].sum(),
        len(patients_clean),
    ],
})
print("Audit summary:")
print(summary.to_string(index=False))

patients_clean = patients_clean[[
    "patient_id", "first_name", "last_name", "dob", "gender", "region",
    "postal_code", "insurance_type", "chronic_diabetes", "chronic_hypertension",
    "registration_date", "flag_no_dob", "flag_no_gender", "flag_no_postal_code",
]]
print("\npatients_clean shape:", patients_clean.shape)
patients_clean.dtypes

Audit summary:
                                            step  count
                                        Raw rows 105000
             Duplicate patient records (dropped)   5000
                Missing dob (kept, tracked only)   1792
     Missing/Unknown gender (kept, tracked only)     64
Missing/invalid postal_code (kept, tracked only)   3661
                       Unique patients remaining 100000

patients_clean shape: (100000, 14)


patient_id                      object
first_name                      object
last_name                       object
dob                     datetime64[ns]
gender                          object
region                          object
postal_code                     object
insurance_type                  object
chronic_diabetes                  bool
chronic_hypertension              bool
registration_date       datetime64[ns]
flag_no_dob                       bool
flag_no_gender                    bool
flag_no_postal_code               bool
dtype: object

In [29]:
# ============================================================
# Clean departments.csv
# ============================================================
# Already clean: 50 rows, no missing values, no duplicate department_id,
# single consistent ID format. Confirm that empirically rather than assume it.

departments_raw = pd.read_csv("departments.csv")
departments_clean = departments_raw.copy()

print(f"Rows: {len(departments_clean):,}")
print(f"Total nulls: {departments_clean.isna().sum().sum()}")
print(f"Duplicate department_id: {departments_clean['department_id'].duplicated().sum()}")
print(f"department_id length values: {departments_clean['department_id'].str.len().unique()}")
print(f"Negative values in numeric columns: {(departments_clean.select_dtypes('number') < 0).sum().sum()}")

departments_clean.head()

Rows: 50
Total nulls: 0
Duplicate department_id: 0
department_id length values: [4]
Negative values in numeric columns: 0


,department_id,department_name,location,annual_budget,base_wait_days,manual_workload_multiplier,claim_denial_risk,lab_delay_risk
0,D001,Cardiology,Main Campus,2494683.86,3,1.70,0.214,0.200
1,D002,Oncology,East Wing,4388388.17,21,1.79,0.104,0.248
2,D003,Neurology,Main Campus,3995981.05,17,0.98,0.060,0.104
3,D004,Emergency,Satellite Clinic,2112350.96,3,1.38,0.041,0.203
4,D005,Radiology,West Wing,3932157.40,30,1.32,0.169,0.081


In [30]:
# ============================================================
# Clean providers.csv
# ============================================================
# Already clean: 2,000 rows, no missing values, no duplicate provider_id,
# single consistent ID format, specialty values match departments.csv
# naming with no casing/typo variants. Confirm empirically, same as
# departments.csv.

providers_raw = pd.read_csv("providers.csv")
providers_clean = providers_raw.copy()

print(f"Rows: {len(providers_clean):,}")
print(f"Total nulls: {providers_clean.isna().sum().sum()}")
print(f"Duplicate provider_id: {providers_clean['provider_id'].duplicated().sum()}")
print(f"provider_id length values: {providers_clean['provider_id'].str.len().unique()}")
print(f"department_id length values: {providers_clean['department_id'].str.len().unique()}")
print(f"Negative values in numeric columns: {(providers_clean.select_dtypes('number') < 0).sum().sum()}")
print(f"years_experience range: {providers_clean['years_experience'].min()}-{providers_clean['years_experience'].max()}")

providers_clean.head()

Rows: 2,000
Total nulls: 0
Duplicate provider_id: 0
provider_id length values: [8]
department_id length values: [4]
Negative values in numeric columns: 0
years_experience range: 1-35


,provider_id,department_id,specialty,years_experience,provider_panel_size,avg_visit_duration_min,overbooking_index
0,PR000001,D016,Obstetrics,25,1244,28.7,2.01
1,PR000002,D034,Dermatology,10,315,30.5,0.89
2,PR000003,D024,Infectious Disease,5,705,40.6,1.21
3,PR000004,D005,Radiology,9,956,32.0,1.43
4,PR000005,D002,Oncology,1,825,24.6,2.34


In [31]:
# ============================================================
# Clean claims.csv
# ============================================================
claims_raw = pd.read_csv("claims.csv")
claims_clean = claims_raw.copy()

print(f"Raw rows: {len(claims_clean):,}")
claims_clean.head()

Raw rows: 505,000


,claim_id,appointment_id,patient_id,department_id,claim_amount,insurance_type_at_claim,insurance_paid,patient_paid,claim_status,submission_date,payment_date,claim_denied_final
0,C000000001,A000179464,P0007209,D027,363.89,Private,0.00,0.00,Pending,2025-08-24,NaN,0
1,C000000002,A000129447,P0093567,D042,1023.43,Private,582.39,441.04,Paid,2024-01-13,2024-03-06,0
2,C000000003,A000204780,P0035732,D013,117.31,Employer,66.76,50.55,Paid,2025-12-04,2025-12-28,0
3,C000000004,A000721527,P0012635,D003,549.19,Public,322.62,226.57,Paid,2024-06-04,2024-07-31,0
4,C000000005,A000364435,P0016099,D049,328.85,Public,289.36,39.49,Paid,2025-05-27,2025-07-08,0


In [32]:
# 1. Exact content-duplicate rows.
# 1,750 rows have a claim_id literally labeled 'C-DUP-####' -- an explicit
# marker that these are duplicates. Checked: 1,750/1,773 of them
# match a normal-looking claim_id row on every field (amount, patient,
# dates, status) when joined on appointment_id. Rather than special-case
# just the labeled ones, drop ANY row that's a full duplicate of another
# on every column except claim_id (claim_id is regenerated per export, so
# it isn't a reliable de-dup key by itself) -- this catches the labeled
# rows plus a handful of unlabeled ones with identical content.

n_before = len(claims_clean)
claims_clean = claims_clean.drop_duplicates(
    subset=[c for c in claims_clean.columns if c != "claim_id"]
).reset_index(drop=True)
n_dupes_dropped = n_before - len(claims_clean)

print(f"Content-duplicate rows dropped: {n_dupes_dropped:,}")
print(f"Rows remaining: {len(claims_clean):,}")
print(f"claim_id unique after dedup: {claims_clean['claim_id'].is_unique}")

Content-duplicate rows dropped: 5,000
Rows remaining: 500,000
claim_id unique after dedup: True


In [33]:
# 2. patient_id -- same 3-format inconsistency as the other tables.
#    Standardize to P + 7 digits.

claims_clean["patient_id"] = (
    claims_clean["patient_id"].str.extract(r"(\d+)$")[0].str.zfill(7)
)
claims_clean["patient_id"] = "P" + claims_clean["patient_id"]

print("patient_id format check after cleaning:")
print(claims_clean["patient_id"].str.len().value_counts())

patient_id format check after cleaning:
patient_id
8    500000
Name: count, dtype: int64


In [34]:
# 3. claim_amount -- 3,029 rows have a NEGATIVE claim_amount, which isn't
#    logically possible (you can't submit a negative claim). Checked: for
#    every OTHER row, (insurance_paid + patient_paid) / claim_amount forms
#    the same clean 0-1 ratio distribution as the negative rows do once you
#    take the absolute value -- so these are sign errors, not real credits/
#    refunds. Fix by taking the absolute value and flag what was corrected.

claims_clean["flag_amount_corrected"] = claims_clean["claim_amount"] < 0
claims_clean["claim_amount"] = claims_clean["claim_amount"].abs()

print(f"claim_amount sign errors corrected: {claims_clean['flag_amount_corrected'].sum():,}")
print(claims_clean["claim_amount"].describe())

claim_amount sign errors corrected: 3,000
count    500000.000000
mean        608.773630
std         442.105753
min          75.000000
25%         318.010000
50%         492.380000
75%         763.710000
max       13896.180000
Name: claim_amount, dtype: float64


In [35]:
# 4. submission_date / payment_date -- same 4 mixed date formats.
#    payment_date is genuinely NaT (not a data-quality issue) whenever a
#    claim hasn't been paid yet: it's ~99% missing for Denied/Pending/Manual
#    Review claims and only ~4% missing for Paid/Complete claims, which
#    tracks with how billing actually works. Left as NaT, not flagged.

claims_clean["submission_date"] = parse_multi_format(claims_clean["submission_date"])
claims_clean["payment_date"] = parse_multi_format(claims_clean["payment_date"])

print(f"Missing submission_date: {claims_clean['submission_date'].isna().sum():,}")
print(f"Missing payment_date: {claims_clean['payment_date'].isna().sum():,} "
      f"(expected -- unpaid claims have no payment date)")

Missing submission_date: 19,856
Missing payment_date: 147,411 (expected -- unpaid claims have no payment date)


In [36]:
# 5. claim_status -- 17 raw values become 4 real statuses once you fix
#    casing, spacing/underscore variants, and synonyms ('Reject'/'Rejected'
#    -> 'Denied', 'Complete' -> 'Paid', 'In Progress' -> 'Pending').
#    Validated against claim_denied_final: every status that maps to "Denied"
#    has claim_denied_final == 1 on 100% of rows, and every other status has
#    claim_denied_final == 0 on 100% of rows -- strong confirmation the
#    mapping is correct.

status_map = {
    "paid": "Paid", "complete": "Paid",
    "denied": "Denied", "reject": "Denied", "rejected": "Denied",
    "manual review": "Manual Review", "manual_review": "Manual Review", "manual": "Manual Review",
    "pending": "Pending", "in progress": "Pending",
}
claims_clean["claim_status"] = (
    claims_clean["claim_status"].str.strip().str.lower().map(status_map)
)

print("claim_status value counts after mapping:")
print(claims_clean["claim_status"].value_counts(dropna=False))

agreement = (
    (claims_clean["claim_status"] == "Denied") == claims_clean["claim_denied_final"].astype(bool)
).mean()
print(f"\nclaim_status='Denied' vs claim_denied_final agreement: {agreement:.2%}")

claim_status value counts after mapping:
claim_status
Paid             366409
Denied            76164
Manual Review     33927
Pending           23500
Name: count, dtype: int64

claim_status='Denied' vs claim_denied_final agreement: 100.00%


In [37]:
# 6. Flag missing department_id.
#    Sparse (~0.3%) and not actually needed for the join -- department context
#    is already available via appointments_clean -- so it's tracked, not excluded.

claims_clean["flag_no_department"] = claims_clean["department_id"].isna()
print(f"Missing department_id: {claims_clean['flag_no_department'].sum():,}")

Missing department_id: 1,467


In [38]:
# 7. Pre-merge check: appointment_id fan-out + audit summary
#    No rows are excluded here -- patient_id and appointment_id are complete
#    for every row, which are the only required join keys. Some appointment_ids
#    still have more than one distinct claim after dedup (legitimate: resubmissions,
#    adjustments) -- not dropped, but worth knowing before joining on appointment_id.

dupe_appts = claims_clean["appointment_id"].duplicated().sum()
print(f"appointment_id values with more than one distinct claim: {dupe_appts:,}")

summary = pd.DataFrame({
    "step": [
        "Raw rows",
        "Content-duplicate rows (dropped)",
        "claim_amount sign errors (kept, corrected)",
        "Missing department_id (kept, tracked only)",
        "Clean rows remaining",
    ],
    "count": [
        len(claims_raw),
        n_dupes_dropped,
        claims_clean["flag_amount_corrected"].sum(),
        claims_clean["flag_no_department"].sum(),
        len(claims_clean),
    ],
})
print("\nAudit summary:")
print(summary.to_string(index=False))

claims_clean = claims_clean[[
    "claim_id", "appointment_id", "patient_id", "department_id",
    "claim_amount", "insurance_type_at_claim", "insurance_paid", "patient_paid",
    "claim_status", "submission_date", "payment_date", "claim_denied_final",
    "flag_amount_corrected", "flag_no_department",
]]
print("\nclaims_clean shape:", claims_clean.shape)
claims_clean.dtypes

appointment_id values with more than one distinct claim: 3,802

Audit summary:
                                      step  count
                                  Raw rows 505000
          Content-duplicate rows (dropped)   5000
claim_amount sign errors (kept, corrected)   3000
Missing department_id (kept, tracked only)   1467
                      Clean rows remaining 500000

claims_clean shape: (500000, 14)


claim_id                           object
appointment_id                     object
patient_id                         object
department_id                      object
claim_amount                      float64
insurance_type_at_claim            object
insurance_paid                    float64
patient_paid                      float64
claim_status                       object
submission_date            datetime64[ns]
payment_date               datetime64[ns]
claim_denied_final                  int64
flag_amount_corrected                bool
flag_no_department                   bool
dtype: object

In [39]:
# ============================================================
# Merge readiness check (all six tables)
# ============================================================
# Confirm join keys share the same format AND actually overlap across all
# six cleaned tables before merging -- matching formats alone isn't enough
# if the underlying ID values don't overlap.

print("patient_id length   -- feedback:", df_clean["patient_id"].str.len().unique(),
      "| patients:", patients_clean["patient_id"].str.len().unique(),
      "| appointments:", appointments_clean["patient_id"].str.len().unique(),
      "| claims:", claims_clean["patient_id"].str.len().unique())

print("department_id length -- feedback:", df_clean["department_id"].dropna().str.len().unique(),
      "| appointments:", appointments_clean["department_id"].str.len().unique(),
      "| departments:", departments_clean["department_id"].str.len().unique(),
      "| providers:", providers_clean["department_id"].str.len().unique(),
      "| claims:", claims_clean["department_id"].dropna().str.len().unique())

print("appointment_id length -- feedback:", df_clean["appointment_id"].str.len().unique(),
      "| appointments:", appointments_clean["appointment_id"].str.len().unique(),
      "| claims:", claims_clean["appointment_id"].str.len().unique())

print("provider_id length   -- appointments:", appointments_clean["provider_id"].dropna().str.len().unique(),
      "| providers:", providers_clean["provider_id"].str.len().unique())

print()
print(f"feedback patient_id found in patients_clean:         {df_clean['patient_id'].isin(patients_clean['patient_id']).mean():.2%}")
print(f"feedback appointment_id found in appointments_clean: {df_clean['appointment_id'].isin(appointments_clean['appointment_id']).mean():.2%}")
print(f"appointments patient_id found in patients_clean:     {appointments_clean['patient_id'].isin(patients_clean['patient_id']).mean():.2%}")
print(f"appointments department_id found in departments_clean: {appointments_clean['department_id'].isin(departments_clean['department_id']).mean():.2%}")
print(f"appointments provider_id found in providers_clean:   {appointments_clean['provider_id'].dropna().isin(providers_clean['provider_id']).mean():.2%}")
print(f"claims appointment_id found in appointments_clean:   {claims_clean['appointment_id'].isin(appointments_clean['appointment_id']).mean():.2%}")
print(f"claims patient_id found in patients_clean:           {claims_clean['patient_id'].isin(patients_clean['patient_id']).mean():.2%}")

print()
print("All six cleaned tables are ready to merge:")
print("  df_clean (feedback)  :", df_clean.shape)
print("  appointments_clean   :", appointments_clean.shape)
print("  patients_clean       :", patients_clean.shape)
print("  departments_clean    :", departments_clean.shape)
print("  providers_clean      :", providers_clean.shape)
print("  claims_clean         :", claims_clean.shape)

patient_id length   -- feedback: [8] | patients: [8] | appointments: [8] | claims: [8]
department_id length -- feedback: [4] | appointments: [4] | departments: [4] | providers: [4] | claims: [4]
appointment_id length -- feedback: [10] | appointments: [10] | claims: [10]
provider_id length   -- appointments: [8] | providers: [8]

feedback patient_id found in patients_clean:         100.00%
feedback appointment_id found in appointments_clean: 99.01%
appointments patient_id found in patients_clean:     100.00%
appointments department_id found in departments_clean: 100.00%
appointments provider_id found in providers_clean:   100.00%
claims appointment_id found in appointments_clean:   99.31%
claims patient_id found in patients_clean:           100.00%

All six cleaned tables are ready to merge:
  df_clean (feedback)  : (289889, 10)
  appointments_clean   : (987211, 11)
  patients_clean       : (100000, 14)
  departments_clean    : (50, 8)
  providers_clean      : (2000, 7)
  claims_clean  

In [40]:
# ============================================================
# MERGE: build the analysis-ready master table
# ============================================================
# Each row represents one feedback submission. Information from the other
# datasets is added using left joins, so all feedback records are kept even
# when related information is unavailable. Missing related data appears as
# NaN rather than causing feedback records to be removed.

master = df_clean.copy()
n_base = len(master)
print(f"Starting from df_clean: {n_base:,} rows (one row per feedback submission)")

Starting from df_clean: 289,889 rows (one row per feedback submission)


In [41]:
# 1. + appointments_clean (on appointment_id)
# appointment_id is unique in appointments_clean (verified during cleaning),
# so this is a safe one-to-one join -- no fan-out risk.
# patient_id and department_id are dropped from the appointments side first:
# both already exist on master, and department_id agrees 100% between the
# two tables on every row where they match (checked below) -- no conflict,
# just avoiding duplicate/confusing columns.

check = master.merge(
    appointments_clean[["appointment_id", "department_id"]],
    on="appointment_id", how="inner", suffixes=("_fb", "_appt"),
)
agreement = (check["department_id_fb"] == check["department_id_appt"]).mean()
print(f"department_id agreement (feedback vs appointments) on matched rows: {agreement:.2%}")

appt_cols = appointments_clean.drop(columns=["patient_id", "department_id"])
master = master.merge(appt_cols, on="appointment_id", how="left", indicator="_m_appt")

print(f"\nRow count after merge: {len(master):,} (unchanged from base: {len(master) == n_base})")
print(master["_m_appt"].value_counts().rename("appointment match"))

department_id agreement (feedback vs appointments) on matched rows: 100.00%

Row count after merge: 289,889 (unchanged from base: True)
_m_appt
both          287008
left_only       2881
right_only         0
Name: appointment match, dtype: int64


In [42]:
# 2. + patients_clean (on patient_id)
# patient_id is unique in patients_clean (deduped during cleaning) -- safe
# one-to-one join. Every feedback row's patient_id resolves (checked in the
# merge-readiness cell above: 100% coverage), so this should match on all rows.

master = master.merge(patients_clean, on="patient_id", how="left", indicator="_m_pat")

print(f"Row count after merge: {len(master):,} (unchanged from base: {len(master) == n_base})")
print(master["_m_pat"].value_counts().rename("patient match"))

Row count after merge: 289,889 (unchanged from base: True)
_m_pat
both          289889
left_only          0
right_only         0
Name: patient match, dtype: int64


In [43]:
# 3. + providers_clean (on provider_id)
# provider_id is unique in providers_clean -- safe one-to-one join.
# department_id is dropped from the providers side (redundant -- already on
# master, and provider->department is a fixed mapping so nothing new to
# reconcile). Rows with no matched appointment also have no provider_id, so
# their provider fields are NaN by construction, not a merge failure.

prov_cols = providers_clean.drop(columns=["department_id"])
master = master.merge(prov_cols, on="provider_id", how="left", indicator="_m_prov")

print(f"Row count after merge: {len(master):,} (unchanged from base: {len(master) == n_base})")
print(master["_m_prov"].value_counts().rename("provider match"))

Row count after merge: 289,889 (unchanged from base: True)
_m_prov
both          287008
left_only       2881
right_only         0
Name: provider match, dtype: int64


In [44]:
# 4. + departments_clean (on department_id)
# department_id is unique in departments_clean, and every feedback row has
# a known department_id (that was one of the exclusion criteria during
# feedback cleaning) -- expect 100% match.

master = master.merge(departments_clean, on="department_id", how="left", indicator="_m_dept")

print(f"Row count after merge: {len(master):,} (unchanged from base: {len(master) == n_base})")
print(master["_m_dept"].value_counts().rename("department match"))

Row count after merge: 289,889 (unchanged from base: True)
_m_dept
both          289889
left_only          0
right_only         0
Name: department match, dtype: int64


In [45]:
# 5. + claims_clean (on appointment_id) -- AGGREGATED first
# appointment_id is NOT unique in claims_clean (3,802 appointments have more
# than one distinct claim -- resubmissions/adjustments). Merging the raw
# table directly would fan out those feedback rows and quietly inflate row
# counts / bias the at-risk rate. Instead, collapse to one row per
# appointment_id first, THEN merge.

claims_agg = claims_clean.groupby("appointment_id").agg(
    total_claim_amount=("claim_amount", "sum"),
    total_patient_paid=("patient_paid", "sum"),
    total_insurance_paid=("insurance_paid", "sum"),
    n_claims=("claim_id", "count"),
    any_claim_denied=("claim_denied_final", "max"),
).reset_index()

master = master.merge(claims_agg, on="appointment_id", how="left", indicator="_m_claims")

print(f"Row count after merge: {len(master):,} (unchanged from base: {len(master) == n_base})")
print(master["_m_claims"].value_counts().rename("claims match"))
print("\n(no claim record is expected for a large share of visits -- not every "
      "appointment generates a claim -- so a low match rate here is normal, not an error)")

Row count after merge: 289,889 (unchanged from base: True)
_m_claims
left_only     145396
both          144493
right_only         0
Name: claims match, dtype: int64

(no claim record is expected for a large share of visits -- not every appointment generates a claim -- so a low match rate here is normal, not an error)


In [46]:
# 6. Clean up merge scaffolding + final validation

master = master.drop(columns=["_m_appt", "_m_pat", "_m_prov", "_m_dept", "_m_claims"])

assert len(master) == n_base, "Row count changed during merge -- a join fanned out unexpectedly"
assert master["feedback_id"].is_unique, "feedback_id should still be unique after merging in 1:1 / aggregated tables"

print("Merge validation passed: row count preserved, feedback_id still unique.")
print(f"\nmaster shape: {master.shape}")
print(f"\nCoverage by joined table:")
print(f"  appointments  : {master['status'].notna().mean():.1%}")
print(f"  patients      : {master['first_name'].notna().mean():.1%}")
print(f"  providers     : {master['specialty'].notna().mean():.1%}")
print(f"  departments   : {master['department_name'].notna().mean():.1%}")
print(f"  claims        : {master['n_claims'].notna().mean():.1%}")

master.head()

Merge validation passed: row count preserved, feedback_id still unique.

master shape: (289889, 48)

Coverage by joined table:
  appointments  : 99.0%
  patients      : 100.0%
  providers     : 99.0%
  departments   : 100.0%
  claims        : 49.8%


,feedback_id,appointment_id,patient_id,department_id,rating,at_risk,complaint_category,comment,feedback_date,patient_satisfaction_after_visit,...,annual_budget,base_wait_days,manual_workload_multiplier,claim_denial_risk,lab_delay_risk,total_claim_amount,total_patient_paid,total_insurance_paid,n_claims,any_claim_denied
0,F000000001,A000052897,P0052751,D038,4,False,Wait Time,I waited much longer than expected.,2024-01-29,Satisfied,...,3442043.06,16,1.58,0.386,0.205,NaN,NaN,NaN,NaN,NaN
1,F000000002,A000679409,P0071824,D007,3,True,Wait Time,The wait was too long.,2025-05-14,At Risk,...,3427269.66,13,1.19,0.208,0.156,NaN,NaN,NaN,NaN,NaN
2,F000000003,A000724365,P0085277,D045,3,True,Communication,Instructions were unclear.,2025-11-11,At Risk,...,2408354.37,40,0.88,0.215,0.228,NaN,NaN,NaN,NaN,NaN
3,F000000004,A000303615,P0029668,D039,3,True,Communication,Communication could be better.,2023-11-24,At Risk,...,2349214.40,38,2.46,0.073,0.200,371.52,30.64,340.88,1.0,0.0
4,F000000005,A000213081,P0014845,D027,3,True,Wait Time,The wait was too long.,2024-02-11,At Risk,...,3348815.28,43,1.13,0.159,0.205,377.46,49.91,327.55,1.0,0.0


In [47]:
# 7. has_claim flag + fill claims columns with structural zeros
# A blank in the claims columns means "no claim was ever filed for this
# visit" -- not "unknown". That's a meaningful zero, not missing data (about
# half of all appointments never generate a claim, which checks out: claims_
# clean has 500,000 rows against appointments_clean's 987,211, a near-
# identical ~50% ratio to what we see here). Add an explicit has_claim flag
# so downstream analysis can filter/segment without relying on NaN checks,
# then fill the claims-derived numeric columns to 0 / False for the no-claim
# rows. complaint_category is NOT touched here -- unlike claims, a blank
# complaint_category has no natural "zero" to fall back on, so it stays NaN
# and should be treated as its own "No category given" bucket in analysis.

master["has_claim"] = master["n_claims"].notna()
print(f"has_claim rate: {master['has_claim'].mean():.1%} "
      f"(matches the earlier claims coverage check)")

claims_numeric_cols = ["total_claim_amount", "total_patient_paid", "total_insurance_paid", "n_claims"]
master[claims_numeric_cols] = master[claims_numeric_cols].fillna(0)
master["any_claim_denied"] = master["any_claim_denied"].fillna(False).astype(bool)

print("\nRemaining nulls in claims columns after fill:")
print(master[claims_numeric_cols + ["any_claim_denied"]].isna().sum())

has_claim rate: 49.8% (matches the earlier claims coverage check)

Remaining nulls in claims columns after fill:
total_claim_amount      0
total_patient_paid      0
total_insurance_paid    0
n_claims                0
any_claim_denied        0
dtype: int64


In [48]:
# 7b  Drop patient-level missingness flags, not needed downstream
master = master.drop(columns=["flag_no_dob", "flag_no_gender", "flag_no_postal_code", "flag_wait_days_corrected"])

print(f"master shape after dropping flags: {master.shape}")

master shape after dropping flags: (289889, 45)


In [49]:
# 8. Save the merged master table
master.to_csv("patient_experience_master.csv", index=False)
print(f"Saved patient_experience_master.csv -- {master.shape[0]:,} rows x {master.shape[1]} columns")
print("\nFinal columns:")
print(list(master.columns))

Saved patient_experience_master.csv -- 289,889 rows x 45 columns

Final columns:
['feedback_id', 'appointment_id', 'patient_id', 'department_id', 'rating', 'at_risk', 'complaint_category', 'comment', 'feedback_date', 'patient_satisfaction_after_visit', 'provider_id', 'scheduled_date', 'appointment_date', 'visit_type', 'wait_days', 'status', 'future_no_show_flag', 'first_name', 'last_name', 'dob', 'gender', 'region', 'postal_code', 'insurance_type', 'chronic_diabetes', 'chronic_hypertension', 'registration_date', 'specialty', 'years_experience', 'provider_panel_size', 'avg_visit_duration_min', 'overbooking_index', 'department_name', 'location', 'annual_budget', 'base_wait_days', 'manual_workload_multiplier', 'claim_denial_risk', 'lab_delay_risk', 'total_claim_amount', 'total_patient_paid', 'total_insurance_paid', 'n_claims', 'any_claim_denied', 'has_claim']
